### PysPark + Iceberg + LangChain + Ollama (Mistral)

In [48]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('rag_chatbot_v1') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [49]:
try:
    from langchain.llms import Ollama
    from langchain.prompts import PromptTemplate
    from dotenv import load_dotenv
    import os
    
    print("Lib já está instalado.")
except ImportError:
    print("Libs não estão instalado. Instalando agora...")
    
    !pip install pyspark langchain
    !pip install langchain_community

Lib já está instalado.


In [34]:
%run ./ConfigEnv.ipynb

python-dotenv já está instalado.


In [35]:
 %run ./Common.ipynb

In [50]:
# config_env('.env', "OLLAMA_API_URL","http://172.18.05:11434/")
load_dotenv()
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
token = os.getenv("API_TOKEN")

In [51]:
%run ./Initialize.ipynb

Dados disponivei: df_posicao


In [41]:
# df_posicao.select('*').show(5)



In [43]:
llm = Ollama(model="mistral:latest", base_url=OLLAMA_API_URL)


# Teste a resposta
response = llm.invoke("Qual é a capital do Brasil?")
print(response)


 A capital do Brasil é Brasília.


In [94]:
spark.sql("""
SELECT SUM(Quantidade_Veiculos) AS Total_Veiculos
FROM tbl_bus_posicao
WHERE Destino_Linha = 'METRÔ JABAQUARA';
""").show()

+--------------+
|Total_Veiculos|
+--------------+
|            13|
+--------------+



In [59]:
from pyspark.sql.functions import explode, col
# from pyspark.sql.types import IntegerType

df = df_posicao.select(
    col('c').alias('Letreiro_Linha'),
    col('cl').alias('Linha'),
    col('sl').alias('Sentido'),
    col('lt0').alias('Destino_Linha'),
    col('lt1').alias('Origem_Linha'),
    col('qv').cast('int').alias('Quantidade_Veiculos')
    
).limit(10)
df.show()
df.createOrReplaceTempView("tbl_bus_posicao")


+--------------+-----+-------+----------------+-----------------+-------------------+
|Letreiro_Linha|Linha|Sentido|   Destino_Linha|     Origem_Linha|Quantidade_Veiculos|
+--------------+-----+-------+----------------+-----------------+-------------------+
|       5318-10|32813|      2|      PÇA. DA SÉ|    CHÁC. SANTANA|                 11|
|       6115-10|33953|      2|    TERM. GRAJAÚ|  CANTINHO DO CÉU|                  8|
|       1788-10|33421|      2|   METRÔ SANTANA|     JD. FONTÁLIS|                  6|
|       177Y-10|33635|      2|       PINHEIROS|METRÔ BARRA FUNDA|                  6|
|       2017-10|  903|      1|      SÃO MIGUEL|        JD. ROBRU|                  6|
|       3027-10|  963|      1|SHOP. ARICANDUVA|  CPTM GUAIANASES|                  6|
|       2733-10|33716|      2|  METRÔ ITAQUERA|      PQ. GUARANI|                  7|
|       8215-10|33342|      2| PÇA. DO CORREIO|   JD. PAULISTANO|                  7|
|       6200-10|33123|      2|  TERM. BANDEIRA| TERM. 

In [88]:
def describe_table(df, table_name="tbl_bus_posicao"):
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"

schema_txt = describe_table(df, "tbl_bus_posicao")

print(schema_txt)


Tabela: tbl_bus_posicao

Colunas:
- Letreiro_Linha: string
- Linha: string
- Sentido: string
- Destino_Linha: string
- Origem_Linha: string
- Quantidade_Veiculos: int


### 4. Inicializar LangChain + Ollama

In [61]:
llm = Ollama(model="mistral:latest", base_url=OLLAMA_API_URL) 


## 5. Criar prompts para SQL e resposta

In [76]:
prompt_sql = PromptTemplate(
    input_variables=["pergunta", "schema"],
    template="""
        Você é um especialista em dados.
        
        Com base na estrutura da tabela abaixo:
        {schema}
        
        Escreva uma consulta SQL (apenas a SQL) para responder à seguinte pergunta:
        {pergunta}
"""
)

prompt_resposta = PromptTemplate(
    input_variables=["pergunta", "resultado"],
    template="""
        Você é um assistente de dados.
        
        Pergunta: {pergunta}
        
        Resultado da consulta:
        {resultado}
        
        Gere uma resposta clara e amigável para o usuário.
"""
)


### 6. Função principal para gerar SQL, executar e responder

In [81]:
def augmented_response(pergunta):
    print(f"Pergunta: {pergunta}")
    
    sql_query = llm.invoke(prompt_sql.format(pergunta=pergunta, schema=schema_txt)).strip()
    print(f"\n SQL Gerado:\n{sql_query}")
    
    try:
        resultado = spark.sql(sql_query).toPandas().to_dict(orient="records")
    except Exception as e:
        print(f"Erro na execução da SQL: {e}")
        return
    
    resposta = llm.invoke(prompt_resposta.format(pergunta=pergunta, resultado=resultado)).strip()
    print(f"\n Resposta:\n{resposta}")

## Augmented Response from Mistral 7B

In [104]:
 augmented_response("Qual o letreiro da linha do veiculo que vai para o metro jabaquara?")

Pergunta: Qual o letreiro da linha do veiculo que vai para o metro jabaquara?

🔍 SQL Gerado:
SELECT Letreiro_Linha
FROM tbl_bus_posicao
WHERE Origem_Linha = 'Não Definido' AND Destino_Linha = 'Metrô Jabaquara';

🤖 Resposta:
Desculpe, mas não tenho informações suficientes sobre letreiros de linhas de ônibus que vão ao Metro Jabaquara. Por favor, tente procurar pelo nome do veículo ou pela empresa responsável por ele no site da prefeitura ou da operadora responsável pelos transportes coletivos na região.


### Modelo Estruturado

In [84]:
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import ChatOllama

In [90]:
llm = ChatOllama(model="mistral:latest", base_url=OLLAMA_API_URL) 

In [99]:
import re

def limpar_sql(resposta_modelo):
    # Remove blocos de código markdown e espaços extras
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql

In [96]:
# Prompt para gerar SQL (com roles)
prompt_sql = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um especialista em dados. Gere apenas a consulta SQL."),
    HumanMessagePromptTemplate.from_template(
        "Com base na estrutura da tabela abaixo:\n\n{schema}\n\n"
        "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}"
    )
])

# Prompt para gerar resposta para o usuário (com roles)
prompt_resposta = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Você é um assistente de dados."),
    HumanMessagePromptTemplate.from_template(
        "Pergunta: {pergunta}\n\nResultado da consulta:\n{resultado}\n\n"
        "Gere uma resposta clara e amigável para o usuário."
    )
])

In [102]:
def augmented_response(pergunta):
    print(f"Pergunta: {pergunta}")

    # Etapa 1: Gerar SQL com role
    sql_chain = prompt_sql | llm
    sql_result = sql_chain.invoke({"pergunta": pergunta, "schema": schema_txt})
    sql_query = limpar_sql(sql_result.content)
    print(f"\n🤖💡 SQL Gerado:\n{sql_query}")

    # Etapa 2: Executar SQL
    try:
        resultado_df = spark.sql(sql_query).toPandas().to_dict(orient="records")
    except Exception as e:
        print(f"❌ Erro na execução da SQL: {e}")
        return

    # Etapa 3: Gerar resposta final com role
    resposta_chain = prompt_resposta | llm
    resposta_result = resposta_chain.invoke({
        "pergunta": pergunta,
        "resultado": resultado_df
    })
    resposta = resposta_result.content.strip()

    print(f"\n🤖 Resposta:\n{resposta}")


In [107]:
augmented_response("Qual o letreiro da linha (Letreiro_Linha) do veiculo que vai para o METRÔ JABAQUARA?")

Pergunta: Qual o letreiro da linha (Letreiro_Linha) do veiculo que vai para o METRÔ JABAQUARA?

🔍 SQL Gerado:
SELECT Letreiro_Linha
   FROM tbl_bus_posicao
   WHERE Destino_Linha = 'METRÔ JABAQUARA';

🤖 Resposta:
O veículo que vai para a estação Metrô Jabaquara possui a designação "695X-10" no seu letreiro da linha.


In [ ]:
def listar_exemplos(self):
    docs = self.qdrant_db.similarity_search("", k=100)
    for i, doc in enumerate(docs):
        score = doc.payload.get("score", 0)  # Acessa o score do payload, caso exista
        print(f"\n🔹 Exemplo {i+1}:")
        print(doc.page_content)
        print(f"🔑 Score de Aprendizado: {score}")